In [19]:
import numpy as np
import random
import os
import json

def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    # 格式: ("文件1", (min1, max1), "文件2", (min2, max2), ...)
    my_recipes = {
        
        # 2源: 两个点源，一个可能很暗，一个可能很亮
        ("./tmp/TYPEA_COMPACT_1000_128_leftup.npy", (0.1, 10.0), "./tmp/TYPEA_COMPACT_1000_128_leftup.npy", (0.1, 10.0)): 20,
        ("./tmp/TYPEA_COMPACT_100_128.npy", (0.1, 10.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 20,
        ("./tmp/TYPEA_COMPACT_100_128.npy", (0.1, 10.0), "./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_COMPACT_100_128.npy", (0.1, 10.0), "./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_COMPACT_100_128.npy", (0.1, 10.0), "./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0)): 10,
        
        ("./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 20,
        ("./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0)): 10,
        
        ("./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_SHELL_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0)): 10,
        
        ("./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0)): 5,
        ("./tmp/TYPEA_DISK_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0)): 5,
        
        ("./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_DIFFUSION_100_128.npy", (10.0, 500.0)): 10,

    }
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=1000, 
        output_image_file="./try/2Sources_1000_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/1000 个样本...
已生成 200/1000 个样本...
已生成 300/1000 个样本...
已生成 400/1000 个样本...
已生成 500/1000 个样本...
已生成 600/1000 个样本...
已生成 700/1000 个样本...
已生成 800/1000 个样本...
已生成 900/1000 个样本...
已生成 1000/1000 个样本...

生成完毕！
图像数据已保存为: ./try/2Sources_1000_128.npy
成分标签已保存为: None


In [36]:
import numpy as np
import random
import os
import json

def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    # 格式: ("文件1", (min1, max1), "文件2", (min2, max2), ...)
    my_recipes = {
        ("./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0)): 35,
        ("./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 10,
        ("./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0)): 35,
        ("./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 5,
        ("./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_20_128.npy", (10.0, 500.0), "./tmp/TYPEA_GAUSSIAN_100_128.npy", (10.0, 500.0)): 5,
    }
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=500, 
        output_image_file="./Multi-sources/MultiGaussian.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/500 个样本...
已生成 200/500 个样本...
已生成 300/500 个样本...
已生成 400/500 个样本...
已生成 500/500 个样本...

生成完毕！
图像数据已保存为: ./Multi-sources/MultiGaussian.npy
成分标签已保存为: None


In [20]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 3):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=1000, 
        output_image_file="./try/3Sources_1000_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/1000 个样本...
已生成 200/1000 个样本...
已生成 300/1000 个样本...
已生成 400/1000 个样本...
已生成 500/1000 个样本...
已生成 600/1000 个样本...
已生成 700/1000 个样本...
已生成 800/1000 个样本...
已生成 900/1000 个样本...
已生成 1000/1000 个样本...

生成完毕！
图像数据已保存为: ./try/3Sources_1000_128.npy
成分标签已保存为: None


In [21]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 4):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=1000, 
        output_image_file="./try/4Sources_1000_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/1000 个样本...
已生成 200/1000 个样本...
已生成 300/1000 个样本...
已生成 400/1000 个样本...
已生成 500/1000 个样本...
已生成 600/1000 个样本...
已生成 700/1000 个样本...
已生成 800/1000 个样本...
已生成 900/1000 个样本...
已生成 1000/1000 个样本...

生成完毕！
图像数据已保存为: ./try/4Sources_1000_128.npy
成分标签已保存为: None


In [22]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 5):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=500, 
        output_image_file="./try/5Sources_500_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/500 个样本...
已生成 200/500 个样本...
已生成 300/500 个样本...
已生成 400/500 个样本...
已生成 500/500 个样本...

生成完毕！
图像数据已保存为: ./try/5Sources_500_128.npy
成分标签已保存为: None


In [24]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 6):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=200, 
        output_image_file="./try/6Sources_200_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/200 个样本...
已生成 200/200 个样本...

生成完毕！
图像数据已保存为: ./try/6Sources_200_128.npy
成分标签已保存为: None


In [25]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 8):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=200, 
        output_image_file="./try/8Sources_200_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/200 个样本...
已生成 200/200 个样本...

生成完毕！
图像数据已保存为: ./try/8Sources_200_128.npy
成分标签已保存为: None


In [26]:
import numpy as np
import random
import os
import json
from itertools import combinations_with_replacement


def generate_complex_sky_dataset(recipes, total_samples, output_image_file, output_label_file, shape=(1, 128, 128)):
    """
    根据带独立通量缩放范围的配方，生成复杂天区数据集及对应标签。
    """
    recipe_list = list(recipes.keys())
    weights = list(recipes.values())
    
    # 1. 解析所有需要用到的独立文件名，加载到缓存
    dataset_cache = {}
    unique_files = set()
    for recipe in recipe_list:
        # 使用步长为 2 的切片 [0::2] 提取所有文件名 (索引 0, 2, 4...)
        for file_name in recipe[0::2]:
            unique_files.add(file_name)
            
    for file_name in unique_files:
        if not os.path.exists(file_name):
            print(f"警告: 找不到文件 {file_name}，已使用零矩阵占位！")
            dataset_cache[file_name] = np.zeros((100, shape[0], shape[1], shape[2]), dtype=np.float32)
        else:
            data = np.load(file_name)
            if data.ndim == 3:
                data = np.expand_dims(data, axis=1)
            dataset_cache[file_name] = data
            
    print("所有基础数据集加载完毕，开始叠加...\n")

    final_dataset = np.zeros((total_samples, shape[0], shape[1], shape[2]), dtype=np.float32)
    
    # 用于保存每一张生成图像的详细物理标签
    all_metadata = []

    # 2. 按照权重随机抽取
    selected_recipes = random.choices(recipe_list, weights=weights, k=total_samples)

    for i, current_recipe in enumerate(selected_recipes):
        canvas = np.zeros(shape, dtype=np.float32)
        
        # 记录当前这张图包含的所有源信息
        image_metadata = {
            "image_id": i,
            "sources": []
        }
        
        # 将 recipe 解析为 (文件名, 范围) 的对偶形式
        # current_recipe[0::2] 是文件名列表，current_recipe[1::2] 是对应的 (min, max) 列表
        files = current_recipe[0::2]
        ranges = current_recipe[1::2]
        
        for source_file, flux_range in zip(files, ranges):
            source_data = dataset_cache[source_file]
            
            # 随机抽取单张图
            idx = random.randint(0, source_data.shape[0] - 1)
            single_image = source_data[idx].copy()
            
            # 应用该源专属的随机亮度扰动因子
            flux_scale = np.random.uniform(flux_range[0], flux_range[1])
            single_image = single_image * flux_scale
            
            # 线性叠加
            canvas += single_image
            
            # 记录该源的信息
            image_metadata["sources"].append({
                "type": source_file.replace(".npy", ""),
                "original_index": idx,
                "applied_flux_scale": round(flux_scale, 4)
            })
            
        final_dataset[i] = canvas
        all_metadata.append(image_metadata)
        
        if (i + 1) % 100 == 0 or (i + 1) == total_samples:
            print(f"已生成 {i + 1}/{total_samples} 个样本...")

    # 3. 保存图像数据和对应的标签字典
    np.save(output_image_file, final_dataset)
    
    # 将标签保存为 JSON 格式，方便后续 PyTorch/TensorFlow 的 Dataset 类读取
    if output_label_file is not None:
        with open(output_label_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=4)
        
    print(f"\n生成完毕！")
    print(f"图像数据已保存为: {output_image_file}")
    print(f"成分标签已保存为: {output_label_file}")

# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    SOURCE_TYPES = {
    "./tmp/TYPEA_COMPACT_100_128.npy": (0.1, 10.0),
    "./tmp/TYPEA_GAUSSIAN_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_SHELL_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DISK_100_128.npy": (10.0, 500.0),
    "./tmp/TYPEA_DIFFUSION_100_128.npy": (10.0, 500.0),
    }

    my_recipes = {}

    for combo in combinations_with_replacement(
            SOURCE_TYPES.keys(), 10):

        recipe = []

        for source in combo:
            recipe.extend([
                source,
                SOURCE_TYPES[source]
            ])

        my_recipes[tuple(recipe)] = 10
    
    generate_complex_sky_dataset(
        recipes=my_recipes, 
        total_samples=100, 
        output_image_file="./try/10Sources_100_128.npy",
        output_label_file=None
    )

所有基础数据集加载完毕，开始叠加...

已生成 100/100 个样本...

生成完毕！
图像数据已保存为: ./try/10Sources_100_128.npy
成分标签已保存为: None
